# 🏛️ Train Virgo on Colab or Kaggle

The same training as on your own GPU server (`scripts/train_local.sh`), in one notebook. Pick what to train
below, then **Run all**. Everything is saved to your private Hugging Face account, where the Virgo server
picks it up on its next start.

| `TRAIN` | What | GPU it needs | Time (rough) |
|---|---|---|---|
| `bayon` | Virgo-1.0-Bayon (chat). 27B on a 24 GB+ GPU (L4, A100), else 12B → Virgo-1.0-Bayon-12B | T4 works (12B) | 1–5 h |
| `angkor` | Virgo-1.0-Angkor-12B (chat) | T4 works | 1–4 h |
| `hearing` | Virgo-1.0-Angkor-Hearing (Khmer speech to text). All of Whisper on 20 GB+, a LoRA add-on on a T4 | T4 works | 2–5 h |
| `voice` | Virgo-1.0-Angkor-Voice (VoxCPM2 fine-tune of Virgo's voice) | T4 works, L4/A100 faster | 1–3 h |
| `voice bayon` | Virgo-1.0-Bayon-Voice: Bayon's own voice, made from a description the first time | T4 works, L4/A100 faster | 1–3 h |

**Before you start**
- **GPU:** Colab → Runtime → Change runtime type → GPU (T4 free; L4/A100 with Colab Pro). Kaggle → Settings → Accelerator → GPU T4 or P100, and Internet on.
- **Hugging Face token** (write access) as a secret named `HF_TOKEN`: Colab 🔑 Secrets (allow notebook access), Kaggle → Add-ons → Secrets.
- **Gemma licence:** accept it once on Hugging Face for [gemma-3-12b-it](https://huggingface.co/google/gemma-3-12b-it) and [gemma-3-27b-it](https://huggingface.co/google/gemma-3-27b-it).
- **Kaggle limits:** sessions stop after 12 h, and 27B (~55 GB download) is too big for Kaggle's disk: train Bayon 12B there, or 27B on Colab with an A100.


In [ ]:
# ⚙️ What to train
TRAIN = "bayon"   # "bayon", "bayon 12b", "angkor", "hearing", "voice", "voice bayon"
EXTRA = ""        # optional, e.g. "--epochs 2", or for voice bayon: set VOICE_DESCRIPTION below
VOICE_DESCRIPTION = ""  # voice bayon only, e.g. "A man in his thirties, deep calm voice, slow and clear"


In [ ]:
# 1️⃣ Set up: GPU, code, Hugging Face login
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: Colab → Runtime → Change runtime type → GPU (Kaggle: Settings → Accelerator → GPU)"
KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
print("✅ GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")

os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"], check=True)
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])

def secret(name):
    if KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    from getpass import getpass
    return os.environ.get(name) or getpass(f"{name}: ").strip()

from huggingface_hub import login, HfApi
token = secret("HF_TOKEN")
login(token=token)
os.environ["HF_TOKEN"] = token
print("✅ Hugging Face:", HfApi().whoami()["name"])


In [ ]:
# 2️⃣ Train (the output streams here; the full log is also kept in the data folder's logs/)
env = {**os.environ}
if VOICE_DESCRIPTION.strip():
    env["VIRGO_VOICE_DESCRIBE"] = VOICE_DESCRIPTION.strip()
command = ["bash", "scripts/train_local.sh", *TRAIN.split(), *EXTRA.split()]
print("▶️", " ".join(command))
with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end="")
if p.returncode:
    raise SystemExit(f"❌ Training stopped (exit {p.returncode}): send the last lines above to Claude.")
print("✅ Done: restart the Virgo server to use it.")
